Biomarkers and prescriptions EDA (reusable version)

Uses the adyn_eda package next to this notebook. SQL lives in queries.py, cleaning in clean.py, analysis in analysis.py, plots in plots.py. Every analysis function returns a DataFrame and every chart function returns a Figure, so the same calls can feed a dashboard later.

Staging data is algorithmically generated, so the numbers show what the pipeline can do, not real findings. Share aggregate outputs only, and clear outputs before committing.

In [ ]:
%load_ext autoreload
%autoreload 2
from eda_biomarker_pkg import Database, AdynData, queries, analysis, plots

db = Database()          
data = AdynData(db)    

1. Overlap between labs and prescriptions

In [ ]:
db.query(queries.OVERLAP)

analysis.products_summary(data.rx_orders)

plots.patients_per_product(analysis.products_summary(data.rx_orders))

2. Hormone coverage, distributions and data quality

In [ ]:
analysis.hormone_summary(data.hormones).round(2)

analysis.placeholder_counts(data.hormones_raw)

plots.hormone_histograms(data.hormones)

spikes = analysis.spike_summary(data.hormones)
spikes

# How many of each sample's hormones equal the spike value (6 = whole sample)
analysis.spike_samples(data.hormones)["n_spiked_hormones"].value_counts()

plots.spike_timeline(data.hormones, spikes)

3. Genetic scores in pipeline output

In [ ]:
db.query(queries.PIPELINE_KEYS)
db.query(queries.GENETIC_FIELDS)
data.genetics.describe().round(2)

4. Prescriptions by hormone level

In [ ]:
rates = analysis.rate_by_thirds(data.patients, data.hormone_names)
rates.round(2)

plots.rate_by_thirds_chart(rates)

tests = analysis.hormone_tests(data.patients, data.hormone_names)
print("Bonferroni cutoff:", round(tests.attrs["bonferroni_cutoff"], 4))
tests.round(3)

# Do hormones / depression score differ by the first product type patients chose?
vars_ = data.hormone_names + ["dep_zscore"]
by_type = analysis.compare_by_type(data.patients, vars_)
print("patients per type:", by_type.attrs["n_per_type"])
print("Bonferroni cutoff:", round(by_type.attrs["bonferroni_cutoff"], 4))
by_type.round(3)

plots.by_type_boxplots(data.patients, vars_)

5. Prescriptions by genetic score

In [ ]:
analysis.rate_by_thirds(data.patients, ["dep_zscore"]).round(2)

analysis.rate_by_group(data.patients, "clot_carrier").round(2)

plots.score_by_outcome(data.patients, "dep_zscore", xlabel="depression risk z-score")

# Among patients with a prescription: do clotting-variant carriers get estrogen methods less often?
rx_only = data.patients[data.patients["any_rx"]].dropna(subset=["clot_carrier"])
analysis.rate_by_group(rx_only, "clot_carrier", outcome="ever_estrogen").round(2)

# Clot-variant carriers vs product type
carriers = analysis.type_by_flag(data.patients, "clot_carrier")
print("chi-square p:", round(carriers.attrs["p_value"], 3))
carriers

6. Product types and switching

In [ ]:
data.rx_patients["switched"].mean().round(2), data.rx_patients["n_types_tried"].value_counts()

plots.heatmap(analysis.switch_matrix(data.rx_patients), "First vs last method type", "last", "first")

# Do hormones or genetics relate to switching? (patients with a prescription only)
rx_pat = data.patients[data.patients["any_rx"]]
analysis.rate_by_thirds(rx_pat, data.hormone_names + ["dep_zscore"], outcome="switched").round(2)

7. Medical biography answers vs prescriptions

In [ ]:
multi = ["bc_goals", "bc_history", "concerns_about_menstrual_cycle",
         "experienced_previously", "side_effects_to_avoid", "family_history"]
for f in multi:
    print(f"\n{f}")
    display(analysis.options_by_type(data.medbio_answers, data.rx_patients, f).round(3))

8. North Star

Valid DNA sample + valid blood sample + completed medical biography. Run the checks first: queries.SAMPLES assumes orders has customer_id, and north_star() assumes status 'valid' and completion == 100.

In [ ]:
# North Star: valid DNA + valid blood + medical biography, and who has a report
ns = analysis.north_star(data.samples, data.medbio_submissions, data.reports)
counts = analysis.north_star_counts(ns)
display(counts)

# Funnel in order
steps = counts[["valid_dna", "valid_blood", "medbio_complete", "north_star", "missing_report"]]
steps.index = ["valid DNA sample", "valid blood sample", "medical biography",
               "all three (North Star)", "North Star but no report"]
display(plots.funnel(steps, title="North Star: patients meeting each condition"))

# Reports without the North Star: which condition is missing?
display(ns[ns["has_report"] & ~ns["north_star"]][["valid_dna", "valid_blood", "medbio_complete"]].value_counts())

# The patients missing a report: when did their samples reach the lab? (dates only)
missing = ns.index[ns["missing_report"]]   # patient ids: keep in the notebook, don't share
display(data.samples[data.samples["customer_id"].isin(missing)]
        .groupby("kind")["received_by_lab_date"].agg(["min", "max"]))